In [2]:
from airflow.sdk import dag, task
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import Param
from airflow.sdk.definitions.param import ParamsDict
from sqlalchemy import create_engine, VARCHAR, Integer,Date, inspect, text
import os
import pandas as pd
from datetime import datetime
import zipfile
import numpy as np
import requests
from datetime import datetime
from pathlib import Path
import shutil
import boto3
from botocore.config import Config
from urllib.parse import quote_plus


c:\Users\Nelio\anaconda3\Lib\site-packages\airflow\__init__.py:45: RuntimeWarning: Airflow currently can be run on POSIX-compliant Operating Systems. For development, it is regularly tested on fairly modern Linux Distros and recent versions of macOS. On Windows you can run it via WSL2 (Windows Subsystem for Linux 2) or via Linux Containers. The work to add Windows support is tracked via https://github.com/apache/airflow/issues/10388, but it is not a high priority.
  warnings.warn(
2026-05-04T14:35:21.066047Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-05-04T14:35:21.072756Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-05-04T14:35:21.080411Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-05-04T14:35:21.089505Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-05-0

In [ ]:
def local_votacao (path):

    USER = "admin"
    PASSWORD = quote_plus("admin")
    HOST = "137.184.110.201"
    PORT = "5432"
    DB_NAME = "eleicao_RR"
    engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}")
    # engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@db:5432/eleicao_RR")

    colunas_leitura = [
    'dt_geracao', 'hh_geracao', 'aa_eleicao', 'dt_eleicao', 'ds_eleicao',
    'nr_turno', 'sg_uf', 'cd_municipio', 'nm_municipio', 'nr_zona',
    'nr_secao', 'cd_tipo_secao_agregada', 'ds_tipo_secao_agregada',
    'nr_secao_principal', 'nr_local_votacao', 'nm_local_votacao',
    'cd_tipo_local', 'ds_tipo_local', 'ds_endereco', 'nm_bairro', 'nr_cep',
    'nr_telefone_local', 'nr_latitude', 'nr_longitude',
    'cd_situ_local_votacao', 'ds_situ_local_votacao', 'cd_situ_zona',
    'ds_situ_zona', 'cd_situ_secao', 'ds_situ_secao', 'cd_situ_localidade',
    'ds_situ_localidade', 'cd_situ_secao_acessibilidade',
    'ds_situ_secao_acessibilidade', 'qt_eleitor_secao',
    'qt_eleitor_eleicao_federal', 'qt_eleitor_eleicao_estadual',
    'qt_eleitor_eleicao_municipal', 'nr_local_votacao_original',
    'nm_local_votacao_original', 'ds_endereco_locvt_original',
    'id_zona_secao_municipio'
    ]
    
    colunas_num = ['qt_eleitor_secao', 'qt_eleitor_eleicao_federal', 'qt_eleitor_eleicao_estadual','qt_eleitor_eleicao_municipal','cd_municipio']
    colunas_data = ['dt_geracao','dt_eleicao']
    colunas_hora = ['hh_geracao']
    
    # Delete table once before processing all files
    with engine.begin() as conn:
        conn.execute(text('DROP TABLE IF EXISTS local_votacao'))
    print("Tabela local_votacao deletada (ou não existia)")
    
    for arquivo in os.listdir(path):
        full_path = os.path.join(path, arquivo)
        if full_path.endswith('.csv'):
            print(f"Lendo o arquivo: {full_path}")
            df = pd.read_csv(full_path, encoding='latin-1', sep=';',dtype=str, usecols=lambda col: col.strip().lower() in colunas_leitura) #nrows= 10  ,
            df.columns = [col.strip().lower() for col in df.columns]
            for col in colunas_num:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col]).astype('Int64')
            for col in colunas_data:
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col], dayfirst=True, format='%d/%m/%Y', errors='coerce').dt.date
            for col in colunas_hora:
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col], format='%H:%M:%S', errors='coerce').dt.time
            df['aa_eleicao'] = df['aa_eleicao'].astype(str).str.lstrip("0")
            df['cd_municipio'] = df['cd_municipio'].astype(str)
            df['id_zona_secao_municipio'] = df['aa_eleicao'] +  df['nr_turno'] + df['nr_zona'] +  df['nr_secao'] +  df['cd_municipio'] #

            df['id_perfil'] = df['aa_eleicao'] +  df['nr_zona'] +  df['nr_secao'] +  df['cd_municipio']
            df['cd_municipio'] = df['cd_municipio'].astype(str)

            # df = df[df['aa_eleicao'] == '2022'] # Filtra apenas os dados de 2022
            
            print(f"Carregando o arquivo na base de dados: {arquivo}")
            df.to_sql("local_votacao", engine, index=False, if_exists='append')
            print(f"Arquivo Carregado na base de dados: {arquivo}")


    return df #print("Processamento concluído")
local_votacao(path = r'C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\local_votacao\csv')

Tabela local_votacao deletada (ou não existia)
Lendo o arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\local_votacao\csv\eleitorado_local_votacao_2020.csv
Carregando o arquivo na base de dados: eleitorado_local_votacao_2020.csv
